In [ ]:
import os
import socket
from flask import Flask, render_template, jsonify, request, redirect, url_for, session
from datetime import datetime

app = Flask(__name__, static_folder='static', static_url_path='/')
app.secret_key = 'clinic_super_secret_key_2026'

# ==========================================
# 擴充：3 個診間的資料庫
# ==========================================
clinic_status = {
    "clinic_1": {"name": "一診 (梁朝偉 醫師)", "current_number": 15},
    "clinic_2": {"name": "二診 (張學友 醫師)", "current_number": 8},
    "clinic_3": {"name": "三診 (黎明 醫師)", "current_number": 22}
}

carousel_data = [
    {"title": "專業醫療服務團隊", "content": "提供專業的復健治療，幫助您恢復健康生活", "image": "images/placeholder_hero.jpg"}
]

news_data = [
    {"title": "2026春節休診公告", "date": "2026-01-20", "content": "2026年春節期間除夕至初三休診，初四起恢復正常門診。"}
]

# ==========================================
# 前台路由
# ==========================================
@app.route('/')
@app.route('/index.html')
def home():
    return render_template('index.html', carousel=carousel_data, news=news_data)

@app.route('/<page_name>.html')
def serve_html_pages(page_name):
    return render_template(f'{page_name}.html')

@app.route('/api/queue')
def get_queue():
    # 將三個診間的資料整包回傳給前端
    return jsonify({"status": "success", "data": clinic_status})

# ==========================================
# 後台管理路由
# ==========================================
@app.route('/login', methods=['GET', 'POST'])
def login():
    error_msg = ""
    if request.method == 'POST':
        if request.form.get('username') == 'admin_matt' and request.form.get('password') == '50961312Lt@325':
            session['logged_in'] = True
            return redirect(url_for('admin'))
        else:
            error_msg = "<p style='color: #d32f2f; font-weight: bold;'>❌ 帳號或密碼錯誤！</p>"

    return f"""
    <div style="font-family: 'Noto Sans TC', sans-serif; background: #f0f2f5; height: 100vh; display: flex; align-items: center; justify-content: center;">
        <div style="background: #fff; padding: 40px; border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.1); width: 350px; text-align: center;">
            <h2 style="color: #0060a8; margin-bottom: 20px;">巨鼎骨科 後台管理</h2>
            {error_msg}
            <form method="POST" style="text-align: left;">
                <label style="color: #555;">帳號：</label>
                <input type="text" name="username" required style="width: 100%; margin: 8px 0 20px; padding: 10px; border: 1px solid #ccc; border-radius: 4px;">
                <label style="color: #555;">密碼：</label>
                <input type="password" name="password" required style="width: 100%; margin: 8px 0 20px; padding: 10px; border: 1px solid #ccc; border-radius: 4px;">
                <button type="submit" style="width: 100%; padding: 12px; background: #0060a8; color: white; border: none; border-radius: 4px; font-size: 16px; cursor: pointer;">安全登入</button>
            </form>
        </div>
    </div>
    """

@app.route('/logout')
def logout():
    session.pop('logged_in', None)
    return redirect(url_for('login'))

@app.route('/admin', methods=['GET', 'POST'])
def admin():
    if not session.get('logged_in'):
        return redirect(url_for('login'))

    if request.method == 'POST':
        action = request.form.get('action')

        # 處理多診間叫號更新
        if action == 'update_queue':
            c1 = request.form.get('clinic_1_number')
            c2 = request.form.get('clinic_2_number')
            c3 = request.form.get('clinic_3_number')
            if c1: clinic_status['clinic_1']['current_number'] = int(c1)
            if c2: clinic_status['clinic_2']['current_number'] = int(c2)
            if c3: clinic_status['clinic_3']['current_number'] = int(c3)

        elif action == 'add_carousel':
            title = request.form.get('title')
            content = request.form.get('content')
            image = request.form.get('image') or 'images/placeholder_hero.jpg'
            if title: carousel_data.insert(0, {"title": title, "content": content, "image": image})

        elif action == 'delete_carousel':
            idx = int(request.form.get('idx'))
            if 0 <= idx < len(carousel_data): carousel_data.pop(idx)

        elif action == 'add_news':
            title = request.form.get('title')
            content = request.form.get('content')
            date_str = datetime.now().strftime("%Y-%m-%d")
            if title: news_data.insert(0, {"title": title, "date": date_str, "content": content})

        elif action == 'delete_news':
            idx = int(request.form.get('idx'))
            if 0 <= idx < len(news_data): news_data.pop(idx)

        return redirect(url_for('admin'))

    # 組合 HTML 字串
    carousel_list_html = "".join([f"<li style='margin-bottom: 10px; padding: 10px; background: #f9f9f9; border-left: 4px solid #0060a8; display: flex; justify-content: space-between; align-items: center;'><span><strong>{item['title']}</strong> - {item['content']}</span><form method='POST' style='display:inline; margin: 0;'><input type='hidden' name='action' value='delete_carousel'><input type='hidden' name='idx' value='{i}'><button type='submit' style='background: #d32f2f; color: white; border: none; padding: 4px 8px; border-radius: 4px; cursor: pointer;'>刪除</button></form></li>" for i, item in enumerate(carousel_data)])
    news_list_html = "".join([f"<li style='margin-bottom: 10px; padding: 10px; background: #f9f9f9; border-left: 4px solid #888; display: flex; justify-content: space-between; align-items: center;'><span>[{item['date']}] <strong>{item['title']}</strong></span><form method='POST' style='display:inline; margin: 0;'><input type='hidden' name='action' value='delete_news'><input type='hidden' name='idx' value='{i}'><button type='submit' style='background: #d32f2f; color: white; border: none; padding: 4px 8px; border-radius: 4px; cursor: pointer;'>刪除</button></form></li>" for i, item in enumerate(news_data)])

    admin_html = f"""
    <div style="font-family: 'Noto Sans TC', sans-serif; background: #f0f2f5; min-height: 100vh; padding: 40px 20px;">
        <div style="max-width: 900px; margin: 0 auto; background: #fff; border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.05); overflow: hidden;">
            
            <div style="background: #0060a8; color: white; padding: 20px 30px; display: flex; justify-content: space-between; align-items: center;">
                <h2 style="margin: 0;">🏥 診所數位管理中心</h2>
                <div>
                    <a href="/" target="_blank" style="color: white; text-decoration: none; margin-right: 20px; font-weight: bold;">👁️ 預覽前台網頁</a>
                    <a href="/logout" style="background: rgba(255,255,255,0.2); padding: 8px 15px; color: white; text-decoration: none; border-radius: 4px;">登出系統</a>
                </div>
            </div>

            <div style="padding: 30px;">
                <div style="background: #f8fcfd; border: 1px solid #e1f0f6; padding: 20px; border-radius: 8px; margin-bottom: 30px;">
                    <h3 style="color: #0060a8; margin-top: 0; margin-bottom: 15px;">📢 多診間叫號管理</h3>
                    <form method="POST" style="display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 15px;">
                        <input type="hidden" name="action" value="update_queue">
                        
                        <div style="background: #fff; padding: 15px; border-radius: 6px; border: 1px solid #ddd; text-align: center;">
                            <h4 style="margin: 0 0 10px;">{clinic_status['clinic_1']['name']}</h4>
                            <div style="font-size: 24px; color: #d32f2f; font-weight: bold; margin-bottom: 10px;">目前: {clinic_status['clinic_1']['current_number']} 號</div>
                            <input type="number" name="clinic_1_number" placeholder="更新號碼" style="width: 100%; padding: 8px; border: 1px solid #ccc; border-radius: 4px;">
                        </div>

                        <div style="background: #fff; padding: 15px; border-radius: 6px; border: 1px solid #ddd; text-align: center;">
                            <h4 style="margin: 0 0 10px;">{clinic_status['clinic_2']['name']}</h4>
                            <div style="font-size: 24px; color: #d32f2f; font-weight: bold; margin-bottom: 10px;">目前: {clinic_status['clinic_2']['current_number']} 號</div>
                            <input type="number" name="clinic_2_number" placeholder="更新號碼" style="width: 100%; padding: 8px; border: 1px solid #ccc; border-radius: 4px;">
                        </div>

                        <div style="background: #fff; padding: 15px; border-radius: 6px; border: 1px solid #ddd; text-align: center;">
                            <h4 style="margin: 0 0 10px;">{clinic_status['clinic_3']['name']}</h4>
                            <div style="font-size: 24px; color: #d32f2f; font-weight: bold; margin-bottom: 10px;">目前: {clinic_status['clinic_3']['current_number']} 號</div>
                            <input type="number" name="clinic_3_number" placeholder="更新號碼" style="width: 100%; padding: 8px; border: 1px solid #ccc; border-radius: 4px;">
                        </div>

                        <button type="submit" style="grid-column: span 3; background: #0060a8; color: white; border: none; padding: 12px; border-radius: 4px; cursor: pointer; font-size: 16px; margin-top: 10px;">💾 同步更新所有診間號碼</button>
                    </form>
                </div>

                <div style="margin-bottom: 40px;">
                    <h3 style="color: #333; border-bottom: 2px solid #eee; padding-bottom: 10px;">🖼️ 首頁輪播圖管理</h3>
                    <form method="POST" style="background: #fafafa; padding: 15px; border-radius: 8px; margin-bottom: 15px; display: grid; gap: 10px;">
                        <input type="hidden" name="action" value="add_carousel">
                        <input type="text" name="title" placeholder="主標題" required style="padding: 8px; border: 1px solid #ccc; border-radius: 4px;">
                        <input type="text" name="content" placeholder="副標題說明" required style="padding: 8px; border: 1px solid #ccc; border-radius: 4px;">
                        <button type="submit" style="background: #28a745; color: white; border: none; padding: 10px; border-radius: 4px; cursor: pointer;">新增輪播項目</button>
                    </form>
                    <ul style="list-style: none; padding: 0; margin: 0;">{carousel_list_html}</ul>
                </div>

                <div>
                    <h3 style="color: #333; border-bottom: 2px solid #eee; padding-bottom: 10px;">📝 最新公告管理</h3>
                    <form method="POST" style="background: #fafafa; padding: 15px; border-radius: 8px; margin-bottom: 15px; display: grid; gap: 10px;">
                        <input type="hidden" name="action" value="add_news">
                        <input type="text" name="title" placeholder="公告標題" required style="padding: 8px; border: 1px solid #ccc; border-radius: 4px;">
                        <textarea name="content" placeholder="公告詳細內容..." required style="padding: 8px; border: 1px solid #ccc; border-radius: 4px; height: 80px;"></textarea>
                        <button type="submit" style="background: #28a745; color: white; border: none; padding: 10px; border-radius: 4px; cursor: pointer;">發布最新公告</button>
                    </form>
                    <ul style="list-style: none; padding: 0; margin: 0;">{news_list_html}</ul>
                </div>
            </div>
        </div>
    </div>
    """
    return admin_html

def find_free_port(start_port=5000):
    port = start_port
    while True:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind(('127.0.0.1', port))
                return port
            except OSError:
                port += 1

if __name__ == '__main__':
    free_port = find_free_port()
    print("=======================================")
    print("✅ 多診間伺服器成功啟動！")
    print(f"👉 前台： http://127.0.0.1:{free_port}/")
    print(f"👉 後台： http://127.0.0.1:{free_port}/admin")
    print("=======================================")
    app.run(host='127.0.0.1', port=free_port, use_reloader=False)

✅ 多診間伺服器成功啟動！
👉 前台： http://127.0.0.1:5001/
👉 後台： http://127.0.0.1:5001/admin
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
127.0.0.1 - - [25/Mar/2026 14:39:21] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [25/Mar/2026 14:39:21] "GET /styles/main.css HTTP/1.1" 200 -
127.0.0.1 - - [25/Mar/2026 14:39:21] "GET /images/doctor1.jpg HTTP/1.1" 200 -
127.0.0.1 - - [25/Mar/2026 14:39:21] "GET /api/queue HTTP/1.1" 200 -
127.0.0.1 - - [25/Mar/2026 14:39:21] "GET /images/placeholder_hero.jpg HTTP/1.1" 200 -
127.0.0.1 - - [25/Mar/2026 14:39:26] "GET /api/queue HTTP/1.1" 200 -
127.0.0.1 - - [25/Mar/2026 14:39:28] "GET /news.html HTTP/1.1" 200 -
127.0.0.1 - - [25/Mar/2026 14:39:28] "GET /styles/main.css HTTP/1.1" 304 -
127.0.0.1 - - [25/Mar/2026 14:39:28] "GET /js/main.js HTTP/1.1" 200 -
127.0.0.1 - - [25/Mar/2026 14:39:30] "GET /about.html HTTP/1.1" 200 -
127.0.0.1 - - [25/Mar/2026 14:39:30] "GET /styles/main.css HTTP/1.1" 304 -
127.0.0.1 - - [25/Mar/2026 14:39:30] "GET /js/main.js HTTP/1.1" 304 -
127.0.0.1 - - [25/Mar/2026 14:39:32] "GET /team.html HTTP/1.1" 200 -
127.0.0